In [0]:

# open the terminal for step 1 and 2
#1. databricks secrets create-scope izgenaiscope
#2. databricks secrets put-secret izgenaiscope databricks_token
DATABRICKS_TOKEN = dbutils.secrets.get(scope="izgenaiscope", key="databricks_token");

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS lakehousecat;
CREATE SCHEMA IF NOT EXISTS lakehousecat.default;
drop table if exists lakehousecat.default.customer_reviews;
CREATE TABLE IF NOT EXISTS lakehousecat.default.customer_reviews (
  review_id STRING,
  city string,
  review_text STRING
);
INSERT INTO lakehousecat.default.customer_reviews VALUES
("REV-001",'NYC', "The shoe are beautiful, but they ripped after two days of wearing them! I want my money back."),
("REV-002",'NY', "The Laptop Shipping took 3 weeks. Absolutely unacceptable."),
("REV-003",'CA', "Perfect fit! Will definitely be buying from you guys again."),
("REV-004",'CAL', "I received the wrong color. I ordered black but got blue. How do I fix this?");


In [0]:
df_reviews = spark.read.table("lakehousecat.default.customer_reviews")
display(df_reviews)

In [0]:
%sql
CREATE OR REPLACE VIEW lakehousecat.default.advanced_text_analysis AS
SELECT 
  review_id,
  review_text,

  -- NLP CONCEPTS: Syntax, Structure, & Mechanics

  -- 1. Tokenization: Breaking text into individual pieces
  ai_query(
    'databricks-meta-llama-3-1-8b-instruct',
    CONCAT('You are a tokenizer. Break the following text down into individual words and punctuation marks. Return ONLY a comma-separated list of tokens: ', review_text)
  ) AS nlp_tokens,

  -- 2. POS (Part-of-Speech) Tagging: Identifying grammatical labels
  ai_query(
    'databricks-meta-llama-3-1-8b-instruct',
    CONCAT('You are a POS tagger. Identify the part of speech for the words in this text. Return ONLY a comma-separated list in "word(TAG)" format (e.g., fast(ADJECTIVE), run(VERB)): ', review_text)
  ) AS nlp_pos_tags,

  -- NLU CONCEPTS: Meaning, Intent, Sentiment & Context

  -- 3. NER (Named Entity Recognition): Extracting specific real-world subjects
  ai_query(
    'databricks-meta-llama-3-1-8b-instruct',
    CONCAT('You are an NER tool., Extract Named Entities such as Person, Organization, Duration, Time, Money, Location, or Product. Return ONLY a comma-separated list in "Entity (Type)" format. If none exist, return "None": ', review_text)
  ) AS nlu_ner_entities,

  -- 4. Sentiment Analysis: Understanding emotional tone
  ai_query(
    'databricks-meta-llama-3-1-8b-instruct',
    CONCAT('You are a sentiment analyzer. Classify the emotional tone of this review. Return ONLY the word POSITIVE, NEGATIVE, or NEUTRAL: ', review_text)
  ) AS nlu_sentiment,

  -- 5. Semantic Matching (Similarity Scoring): Checking for a specific underlying goal (This example is to show the similarity scoring used behind)
  ai_query(
    'databricks-meta-llama-3-1-8b-instruct',
    CONCAT('Does this review express an intent to return the product, ask for a refund, or complain about shipping? Return ONLY the word YES or NO: ', review_text)
  ) AS nlu_return_semantic_matching,

  -- 6. Intent Classification/Generation: Identifying the broad purpose of the user
  ai_query(
    'databricks-meta-llama-3-1-8b-instruct',
    CONCAT('Identify the primary intent of this customer review. Choose ONLY ONE from the following categories: "Product Inquiry", "Complaint", "Praise", "Feature Request", "Customer Support", or "Other". Return ONLY the category name: ', review_text)
  ) AS nlu_primary_intent,

  -- 7. Vector Embeddings: Converting meaning into mathematical representations
  -- Note: Text generation models (like Llama) don't create embeddings. 
  ai_query(
    'databricks-bge-large-en', 
    review_text
  ) AS nlu_vector_values

FROM lakehousecat.default.customer_reviews;

SELECT * FROM lakehousecat.default.advanced_text_analysis;